# 1 load package and data

In [ ]:
import json
from collections import namedtuple, Counter

import numpy as np
import pandas as pd
import scanpy as sc
import pandas as pd
import squidpy as sq

## Image manipulation and geometry
from tifffile import imread
from skimage.io import imread as skimread

## Plotting imports
from matplotlib import pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize, to_hex, Colormap
from matplotlib.cm import ScalarMappable
from matplotlib.colorbar import ColorbarBase
from matplotlib.lines import Line2D
from matplotlib import rc_context

In [ ]:
folder_path = r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\analysis\2st_2025_4_9\backup_adata\QC_10_50_2st\embryo\subcluster"
h5ad_file = Path(folder_path) / "cluster6_hvg_res0.5.h5ad"
c6 = sc.read_h5ad(h5ad_file)

# 2.Extended Data Fig. 6a

In [ ]:

tsne_coords =c6.obsm['X_umap']


leiden_clusters = c6.obs['leiden']

import pandas as pd

tsne_df = pd.DataFrame(tsne_coords, columns=['UMAP1', 'UMAP2'])
tsne_df['leiden'] = leiden_clusters.values

plt.figure(figsize=(6, 4))
sns.scatterplot(data=tsne_df, x='UMAP1', y='UMAP2', hue='leiden', palette=custom_palette, s=6, edgecolor=None)
plt.title("t-SNE Plot with Leiden Clusters")
plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.legend(title="Leiden Cluster", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig(r'P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\paper\3st_2025_1_27\Figure4\umap\umap_clustye6_4st.png', format="png",dpi=900,bbox_inches='tight')
plt.show()

# 3.Extended Data Fig. 6b

In [ ]:
sc.tl.rank_genes_groups(c6, 
                        groupby="leiden", 
                        method="wilcoxon", 
                        pts=True,
                        )

In [ ]:
sc.tl.dendrogram(c6, groupby="leiden", linkage_method="single",cor_method='pearson')  #'pearson', 'spearman', 'kendall',
sc.pl.dendrogram(c6, groupby="leiden")

In [ ]:
my_genes = [
   "TFAP2A","EPCAM","SOX9","ANXA2", "DLK1","H19", "EIF1B", "PAX1",
    "PITX2", "NKX2-1"
]

sc.pl.dotplot(
    c6,
    var_names=my_genes,
    groupby='leiden',  # 你的分组列名
    standard_scale='var',  # 标准化
    #vmax=2, 
    cmap="Reds",
    figsize=(5, 2.5),
    dendrogram=True,
    save='C6_leiden_sub_normalize.pdf'
)

# 4.Extended Data Fig. 6i,j

In [ ]:
import pandas as pd

df = pd.read_csv(r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\analysis\2st_2025_4_9\export for cellNEST\19st_2026_02_27_for_gut_liver\CellNEST_MYDATA_3D_top20percent.csv")

In [ ]:

m = embryo.obs.set_index('cell_id_1')['leiden_hvg_sub']


df['leiden_from'] = df['from_cell'].map(m)



df['leiden_to'] = df['to_cell'].map(m)

df["leiden_from"] = df["leiden_from"].astype(str)
df["leiden_to"] = df["leiden_to"].astype(str)
pd.set_option('display.max_rows', None)
print(df["leiden_to"].value_counts())

In [ ]:
df = df[
    ~((df['leiden_from'].isin(['e17_6.0', 'e17_6.1','e17_6.2', 'e17_1','e17_3', 'e17_0','e17_7','e17_4','e6_4','e6_1','e6_5', 'e6_3', 'e6_0',"e6_6","e3_2","e3_5","e25_1","e25_2"])) | 
      (df['leiden_to'].isin(['e17_6.0', 'e17_6.1','e17_6.2', 'e17_1','e17_3', 'e17_0','e17_7','e17_4','e6_4','e6_1','e6_5', 'e6_3', 'e6_0',"e6_6","e3_2","e3_5","e25_1","e25_2"])))
]

In [ ]:
# 将 leiden_from 和 leiden_to 中以 'e11_' 开头的都替换成 'e11'
df['leiden_from'] = df['leiden_from'].apply(lambda x: 'e11' if str(x).startswith('e11_') else x)
df['leiden_to'] = df['leiden_to'].apply(lambda x: 'e11' if str(x).startswith('e11_') else x)


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# ==========================================
# 0. 准备数据
# ==========================================
real_niche_counts = {
    'e11': 5124, 
    'e3_0': 2223,
    'e3_1': 1546,
    'e3_4': 883,
    'e3_3': 931,
    'e6_2': 964,
    'e25_0': 277,
    'e17_2': 410,
    'e17_5': 135,
    'e6_7': 176
}
counts_series = pd.Series(real_niche_counts)

desired_order = ['e6_2', 'e6_7', 'e17_2', 'e17_5', "e11",'e3_0', "e3_1", "e3_3",'e3_4', 'e25_0']


spec_count_matrix = df.groupby(['leiden_from', 'leiden_to'], observed=True)['attention_score'].count().unstack(fill_value=0)


counts_series = counts_series.reindex(desired_order)
spec_density_matrix = spec_count_matrix.div(counts_series, axis=0)
spec_density_matrix = spec_density_matrix.reindex(index=desired_order, columns=desired_order).fillna(0)


annot_labels = spec_density_matrix.map(lambda x: f"{x:.0f}" if x > 2 else "")


plt.figure(figsize=(5, 5))
sns.heatmap(
    spec_density_matrix, 
    annot=annot_labels,
    fmt='',
    cmap='Reds', 
    linewidths=.5, 
    linecolor='white',
    vmin=0,
    vmax=20,
    cbar_kws={'label': 'Specific Interactions per Sender Cell'},
    annot_kws={'size': 12}
)
plt.title('Normalized Density of Specific Communication\n(Corrected for Cell Abundance)', fontsize=14)
plt.xlabel('Receiver Niche (To)')
plt.ylabel('Sender Niche (From)')
plt.tight_layout()
plt.savefig(r'P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\paper\3st_2025_1_27\Figure4\cellnest\cellnest_midgut_vensal.pdf', dpi=900, bbox_inches='tight')
plt.show()

In [ ]:
def visualize_lr_network(df, from_leiden, to_leiden, top_n=15, 
                        show_scatter=False, save_path=None,
                        figsize=None, dpi=300, 
                        color_by='specificity'): 

    import matplotlib.pyplot as plt
    import seaborn as sns
    from adjustText import adjust_text
    

    pair_df = df[(df['leiden_from'] == from_leiden) & 
                 (df['leiden_to'] == to_leiden)].copy()
    
    if len(pair_df) == 0:

        return
    

    lr_summary = pair_df.groupby(['ligand', 'receptor']).agg({
        'attention_score': ['mean', 'count'],
        'specificity_score': 'mean'  
    }).reset_index()
    lr_summary.columns = ['ligand', 'receptor', 'avg_attention', 'count', 'avg_specificity']
    

    lr_summary = lr_summary.nlargest(top_n, 'count')
    
    if len(lr_summary) == 0:
    
        return
    
    lr_summary = lr_summary.sort_values('count', ascending=True)
    

    if figsize is None:
        if show_scatter:
            figsize = (18, 8)
        else:
            width = 10
            height = max(6, len(lr_summary) * 0.5)
            figsize = (width, height)
    

    if show_scatter:
        fig, axes = plt.subplots(1, 2, figsize=figsize)
        ax1 = axes[0]
    else:
        fig, ax1 = plt.subplots(figsize=figsize)
    
  
    lr_summary['pair'] = lr_summary['ligand'] + ' _ ' + lr_summary['receptor']
    
    if color_by == 'specificity':
      
        color_values = lr_summary['avg_specificity']
        cbar_label = 'Avg Specificity Score'
        global_min = lr_summary['avg_specificity'].min()
        global_max = lr_summary['avg_specificity'].max()
    elif color_by == 'attention':
       
        color_values = lr_summary['avg_attention']
        cbar_label = 'Avg Attention Score'
        global_min = lr_summary['avg_attention'].min()
        global_max = lr_summary['avg_attention'].max()
    else:
       
        color_values = None
    

    if color_values is not None:
        norm = plt.Normalize(vmin=global_min, vmax=global_max)
        cmap = plt.cm.RdYlBu_r  
        colors = cmap(norm(color_values))
        
        bars = ax1.barh(range(len(lr_summary)), lr_summary['count'], 
                        color=colors, edgecolor='black', linewidth=0.5)
        
        # 添加 colorbar
        sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
        sm.set_array([])
        plt.colorbar(sm, ax=ax1, label=cbar_label)
    else:
        bars = ax1.barh(range(len(lr_summary)), lr_summary['count'], 
                        color='steelblue', edgecolor='black', linewidth=0.5)
    
    ax1.set_yticks(range(len(lr_summary)))
    ax1.set_yticklabels(lr_summary['pair'], fontsize=10)
    ax1.set_xlabel('Interaction Count', fontsize=12, fontweight='bold')
    ax1.set_title(f'Top {top_n} L-R Pairs: {from_leiden} → {to_leiden}', 
                  fontsize=14, fontweight='bold')
    #ax1.grid(axis='x', alpha=0.3)
    
  
    if show_scatter:
        ax2 = axes[1]
        
        if color_by == 'specificity':
            scatter = ax2.scatter(lr_summary['count'], lr_summary['avg_specificity'],
                                 s=200, c=lr_summary['avg_specificity'],
                                 cmap=cmap, norm=norm, alpha=0.8,
                                 edgecolors='black', linewidth=1)
            ax2.set_ylabel('Average Specificity Score', fontsize=12)
        else:
            scatter = ax2.scatter(lr_summary['count'], lr_summary['avg_attention'],
                                 s=200, color='steelblue', alpha=0.7,
                                 edgecolors='black', linewidth=1)
            ax2.set_ylabel('Average Attention Score', fontsize=12)
        
        ax2.set_xlabel('Interaction Count', fontsize=12)
        ax2.set_title('L-R Pair Frequency vs Specificity', fontsize=14)
        ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=dpi, bbox_inches='tight')

    
    plt.show()
    
    return lr_summary.sort_values('count', ascending=False)

In [ ]:
result = visualize_lr_network(df, 'e3_1', 'e17_5', top_n=5, 
                              color_by='none',  # 推荐！
                              figsize=(2.5, 2),
                              save_path=r'P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\paper\3st_2025_1_27\Figure4\cellnest\cellnest_midgut_vensal_3_1_17_5.pdf'
                             )

In [ ]:
result = visualize_lr_network(df, 'e3_0', 'e17_2', top_n=5, 
                              color_by='none',  # 推荐！
                              figsize=(2.5, 2),
                              save_path=r'P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\paper\3st_2025_1_27\Figure4\cellnest\cellnest_midgut_vensal_3_0_17_2.pdf'
                             )

In [ ]:
result = visualize_lr_network(df, 'e3_5', 'e17_2', top_n=5, 
                              color_by='none',  # 推荐！
                              figsize=(2.5, 2),
                              save_path=r'P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\paper\3st_2025_1_27\Figure4\cellnest\cellnest_midgut_vensal_3_0_17_2.pdf'
                             )

In [ ]:
result = visualize_lr_network(df, 'e3_3', 'e17_2', top_n=5, 
                              color_by='none',  # 推荐！
                              figsize=(2.5, 2),
                              save_path=r'P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\paper\3st_2025_1_27\Figure4\cellnest\cellnest_midgut_vensal_3_0_17_2.pdf'
                             )

In [ ]:
result = visualize_lr_network(df, 'e17_2', 'e17_5', top_n=5, 
                              color_by='none',  # 推荐！
                              figsize=(2.5, 2),
                              save_path=r'P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\paper\3st_2025_1_27\Figure4\cellnest\cellnest_midgut_vensal_17_2_17_5.pdf'
                             )

In [ ]:
result = visualize_lr_network(df, 'e6_7', 'e6_2', top_n=5, 
                              color_by='none',  # 推荐！
                              figsize=(2.5, 2),
                              save_path=r'P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\paper\3st_2025_1_27\Figure4\cellnest\cellnest_midgut_vensal_6_7_6_2.pdf'
                             )

In [ ]:
result = visualize_lr_network(df, 'e11', 'e6_2', top_n=5, 
                              color_by='none',  # 推荐！
                              figsize=(2.5, 2),
                              save_path=r'P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\paper\3st_2025_1_27\Figure4\cellnest\cellnest_midgut_vensal_6_7_6_2.pdf'
                             )